In [2]:
import geopandas as gpd
import pandas as pd
import json
from pathlib import Path

In [3]:
DATA_DIR = Path.cwd() / 'data'
print(f"{DATA_DIR=}")
INFO_DIR = Path.cwd() / 'info'
print(f"{INFO_DIR=}")
GEOJSON_PATH = Path('/projectnb/planet/PLSP/geojson')
print(f'{GEOJSON_PATH=}')

DATA_DIR=PosixPath('/home/alex-fache/Documents/GitHub/PLSP/code/sites_info/data')
INFO_DIR=PosixPath('/home/alex-fache/Documents/GitHub/PLSP/code/sites_info/info')
GEOJSON_PATH=PosixPath('/projectnb/planet/PLSP/geojson')


In [5]:
# from https://phenocam.nau.edu/webcam/network/table/
phenocam_df = pd.read_csv(DATA_DIR / 'phenocam_site_table.csv')
phenocam_df.rename(columns={c: f'(p){c}' for c in list(phenocam_df.columns)}, inplace=True)
phenocam_df.rename(columns={'(p)site_name': 'phenocam'}, inplace=True)

In [6]:
# from https://ameriflux.lbl.gov/sites/site-search/ export results
flux_df = pd.read_csv(DATA_DIR / 'AmeriFlux-site-search-results-202607161927.tsv', sep='\t')
flux_df.rename(columns={c: f'(f){c}' for c in list(flux_df.columns)}, inplace=True)
flux_df.rename(columns={'(f)Latitude (degrees)': '(f)latitude', '(f)Longitude (degrees)': '(f)longitude'}, inplace=True)

In [7]:
# manual
sites_of_interest_df = pd.read_csv(INFO_DIR / '01_selected_sites_raw.csv')

In [8]:
"""
merge
"""

data_all_df = pd.merge(
    sites_of_interest_df,
    phenocam_df,
    on='phenocam',
    how='left'
)

data_all_df = pd.merge(
    data_all_df.assign(temp_key=data_all_df['site_id'].str.upper()),
    flux_df.assign(temp_key=flux_df['(f)Site ID'].str.upper()),
    on='temp_key',
    how='left'
).drop(columns=['temp_key', '(f)Site ID'])

In [10]:
def get_geojson_file_path(row):
    return GEOJSON_PATH / f'{row["site_name"]}.geojson'


def read_geometry(row):
    
    with open(row['(f)geojson_file_path'], "r") as f:
        geo = json.load(f)
    
        return geo['features']['geometry']['coordinates']


data_all_df['(f)geojson_file_path'] = data_all_df.apply(get_geojson_file_path, axis=1)
# data_all_df['coordinates'] = data_all_df.apply(read_geometry, axis=1)

In [11]:
data_all_df.head()

,site_full,site_id,site_name,phenocam,plsp_number,(p)site_url,(p)latitude,(p)longitude,(p)elevation_m,(p)active,...,(f)AmeriFlux BASE DOI,(f)AmeriFlux FLUXNET Data Start,(f)AmeriFlux FLUXNET Data End,(f)Years of AmeriFlux FLUXNET Data,(f)AmeriFlux FLUXNET DOI,(f)Site Start,(f)Site End,(f)BASE variables available,(f)FLUXNET variables available,(f)geojson_file_path
0,US-ARM_ARM_Southern_Great_Plains_site,US-ARM,ARM_Southern_Great_Plains_site,southerngreatplains,NaN,https://phenocam.nau.edu/webcam/sites/southern...,36.60580,-97.48880,314.0,True,...,https://doi.org/10.17190/AMF/1246027,2003.0,2025.0,"2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010...",https://doi.org/10.17190/AMF/1854366,2002.0,NaN,"CO2, FC, FC_SSITC_TEST, FETCH_70, FETCH_80, FE...",NaN,/projectnb/planet/PLSP/geojson/ARM_Southern_Gr...
1,US-Mpj_Mountainair_Pinyon-Juniper_Woodland,US-Mpj,Mountainair_Pinyon-Juniper_Woodland,usmpj,NaN,https://phenocam.nau.edu/webcam/sites/usmpj/,34.43845,-106.25436,2126.0,True,...,https://doi.org/10.17190/AMF/1246123,2008.0,2025.0,"2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015...",https://doi.org/10.17190/AMF/1832161,2007.0,NaN,NaN,NaN,/projectnb/planet/PLSP/geojson/Mountainair_Pin...
2,US-xKZ_NEON_Konza_Prairie_Biological_Station,US-xKZ,NEON_Konza_Prairie_Biological_Station,konza,NaN,https://phenocam.nau.edu/webcam/sites/konza/,39.08240,-96.56030,443.0,False,...,https://doi.org/10.17190/AMF/1562392,2019.0,2024.0,"2019, 2020, 2021, 2022, 2023, 2024",https://doi.org/10.17190/AMF/1985445,2017.0,NaN,"CH4, CH4_MIXING_RATIO, CO2, CO2C13, CO2_MIXING...",NaN,/projectnb/planet/PLSP/geojson/NEON_Konza_Prai...
3,US-SRG_Santa_Rita_Grassland,US-SRG,Santa_Rita_Grassland,srg,NaN,https://phenocam.nau.edu/webcam/sites/srg/,31.78940,-110.82760,1281.0,True,...,https://doi.org/10.17190/AMF/1246154,2008.0,2025.0,"2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015...",https://doi.org/10.17190/AMF/2204877,2008.0,NaN,"CO2, FC, G, H, H2O, LE, LW_IN, LW_OUT, NETRAD,...",NaN,/projectnb/planet/PLSP/geojson/Santa_Rita_Gras...
4,US-SRM_Santa_Rita_Mesquite,US-SRM,Santa_Rita_Mesquite,srm,NaN,https://phenocam.nau.edu/webcam/sites/srm/,31.82140,-110.86610,1116.0,True,...,https://doi.org/10.17190/AMF/1246104,2004.0,2025.0,"2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011...",https://doi.org/10.17190/AMF/2475756,2004.0,NaN,"CO2, FC, G, H, H2O, LE, LW_IN, LW_OUT, NETRAD,...",NaN,/projectnb/planet/PLSP/geojson/Santa_Rita_Mesq...


In [ ]:
selected_sites_short_info_columns = [
    'site_full', 'site_id', 'site_name', 'plsp_number', '(f)latitude', '(f)longitude', 'phenocam', '(p)latitude', '(p)longitude', 'coordinates'
]

selected_sites_long_info_columns = [
    
]

data_all_df[selected_sites_short_info_columns].to_csv(INFO_DIR / '02_selected_sites_short.csv', index=False)
# data_all_df[selected_sites_long_info_columns].to_csv(INFO_DIR / '03_selected_sites_long.csv', index=False)